# Manufacturing Stress: Protected Evaluation

Limited held-out evaluation. Every successful predictor run consumes one tracked budget unit.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import yaml
from dotenv import load_dotenv

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'uv.lock').exists() and (p / 'implementations').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found.')
for package_root in (REPO_ROOT / 'aieng-forecasting', REPO_ROOT / 'implementations'):
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))
load_dotenv(REPO_ROOT / '.env', override=False)

from aieng.forecasting.evaluation import EvalSpec, EvalTracker, EvalBudgetExceededError, evaluate
from manufacturing_stress_forecasting.data import build_manufacturing_stress_service
from manufacturing_stress_forecasting.hybrid_agent.agent import build_hybrid_agent_predictor
from manufacturing_stress_forecasting.predictors import ManufacturingStressLogisticPredictor
from manufacturing_stress_forecasting.adaptive_agent.agent import build_manufacturing_adaptive_agent_predictor

RUN_PROTECTED_EVAL = False
PREDICTOR_CHOICE = 'logistic'  # logistic, hybrid, adaptive_lite, or adaptive_advanced
SPEC_PATH = REPO_ROOT / 'implementations' / 'manufacturing_stress_forecasting' / 'specs' / 'manufacturing_stress_protected_eval.yaml'
TRACKER_PATH = REPO_ROOT / 'data' / 'predictions' / 'manufacturing_stress_protected_evaluation' / 'eval_runs.yaml'
TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)
eval_spec = EvalSpec.model_validate(yaml.safe_load(SPEC_PATH.read_text(encoding='utf-8')))
tracker = EvalTracker(TRACKER_PATH)
service = build_manufacturing_stress_service()
runs_used = tracker.runs_for(eval_spec.spec_id)
print({'spec_id': eval_spec.spec_id, 'runs_used': runs_used, 'remaining': max(0, (eval_spec.max_runs or 0) - runs_used), 'run': RUN_PROTECTED_EVAL})

In [ ]:
if RUN_PROTECTED_EVAL:
    if PREDICTOR_CHOICE == 'logistic':
        predictor = ManufacturingStressLogisticPredictor()
        trace_status = 'not applicable: deterministic predictor'
    elif PREDICTOR_CHOICE == 'hybrid':
        predictor = build_hybrid_agent_predictor()
        trace_status = 'decision metadata retained; Langfuse tracing is disabled for hybrid'
    elif PREDICTOR_CHOICE == 'adaptive_lite':
        predictor = build_manufacturing_adaptive_agent_predictor(model='gemini-3.1-flash-lite-preview')
        trace_status = 'Langfuse tracing enabled when credentials are configured'
    elif PREDICTOR_CHOICE == 'adaptive_advanced':
        predictor = build_manufacturing_adaptive_agent_predictor(model='gemini-3.5-flash')
        trace_status = 'Langfuse tracing enabled when credentials are configured'
    else:
        raise ValueError(f'Unknown predictor choice: {PREDICTOR_CHOICE}')
    result = evaluate(predictor, eval_spec, service, tracker)
    decision_rows = []
    for prediction, score in zip(result.predictions, result.scores):
        decision = dict(prediction.metadata)
        decision_rows.append({
            'as_of': prediction.as_of,
            'forecast_date': prediction.forecast_date,
            'probability': prediction.payload.probability,
            'brier': score,
            'rationale': decision.get('rationale'),
            'direction': decision.get('direction'),
            'anchor_probability': decision.get('anchor_probability'),
            'agent_probability': decision.get('agent_probability'),
            'adjusted_probability': decision.get('adjusted_probability'),
            'supporting_evidence': decision.get('supporting_evidence'),
            'countervailing_evidence': decision.get('countervailing_evidence'),
        })
    decision_trace = pd.DataFrame(decision_rows)
    print({'predictor_id': result.predictor_id, 'run_number': result.run_number, 'mean_score': result.mean_score, 'trace_status': trace_status})
    display(decision_trace)
else:
    print('RUN_PROTECTED_EVAL = False; no protected evaluation consumed.')

In [ ]:
import matplotlib.pyplot as plt

if 'result' in globals():
    score_frame = pd.DataFrame([{
        'predictor': result.predictor_id,
        'mean_brier': float(result.mean_score),
        'run_number': result.run_number,
    }])
    ax = score_frame.plot.bar(x='predictor', y='mean_brier', legend=False, figsize=(7, 4), color='#2f6f8f')
    ax.set_title('Protected evaluation score')
    ax.set_ylabel('Mean Brier score (lower is better)')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print('Run the protected evaluation to render the score plot.')